In [2]:
import sqlite3
import spacy
import pandas as pd
from collections import Counter

/home/judit/Desktop/repositories/job-market-insights/job-env/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
conn = sqlite3.connect("jobs.db")
cursor = conn.cursor()

In [4]:
query = """
    SELECT * 
    FROM jobspy
    WHERE country='Spain'
    AND id IN (
        SELECT id
        FROM searchterms
        WHERE search_term='data scientist'
        )
"""

df = pd.read_sql(query, conn)

In [5]:
def convert_description(text):
    nlp = spacy.load("en_core_web_lg")
    doc = nlp(text)
    words = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]  # Lemmatization & stopword removal
    return words

In [6]:
df['words_lemma'] = df['description'].apply(lambda x: convert_description(x))

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f682ed20fd0>>
Traceback (most recent call last):
  File "/home/judit/Desktop/repositories/job-market-insights/job-env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f682ed20fd0>>
Traceback (most recent call last):
  File "/home/judit/Desktop/repositories/job-market-insights/job-env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
df.words_lemma

### Single-word counting

In [ ]:
counter = Counter()
for desc in df.words_lemma:
    counter += Counter(desc)

In [ ]:
counter.most_common()

In [ ]:
counter['python']

In [ ]:
counter['gcp']

### Bigram analysis

In [ ]:
from itertools import pairwise

bigram_counts = Counter()
for desc in df.words_lemma:
    pairs = list(pairwise(desc))
    bigram_counts += Counter(pairs)

In [ ]:
bigram_counts.most_common()

In [ ]:
# Contain python
filtered_bigrams = Counter({k: v for k, v in bigram_counts.items() if 'python' in k})

In [ ]:
filtered_bigrams.total()

In [ ]:
words_to_filter = {"python", "english", "german", "aws", "cloud", "azure", "gcp", "r", "go"}  # Words we want to filter for

filtered_bigrams = Counter({k: v for k, v in bigram_counts.items() if any(word in words_to_filter for word in k)})

In [ ]:
filtered_bigrams.most_common()

In [ ]:
import networkx as nx
import plotly.graph_objects as go

G = nx.DiGraph()

for (word1, word2), count in filtered_bigrams.items():
    G.add_edge(word1, word2, weight=count)

pos = nx.spring_layout(G, seed=42)
# Extract edge coordinates
edge_x, edge_y, edge_weights = [], [], []
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_weights.append(edge[2]['weight'])

# Create Plotly edge trace
edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=2, color='cornflowerblue'),
    hoverinfo='text',
    text=[f'Count: {w}' for w in edge_weights],
    mode='lines'
)

# Create node trace
node_x, node_y, node_labels = [], [], []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_labels.append(node)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=node_labels,
    textposition="top center",
    marker=dict(size=15, color='orangered', line=dict(width=2, color='black')),
    hoverinfo='text'
)

# Generate the figure
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title="Bigram Graph Visualization",
    showlegend=False,
    hovermode='closest',
    margin=dict(b=20, l=20, r=20, t=40),
)

fig.show()

In [ ]:
from pyvis.network import Network

# Create a Pyvis network
net = Network(notebook=True, height="500px", width="100%")

# Add nodes and edges
for (word1, word2), count in filtered_bigrams.items():
    word1_tot = Counter({k: v for k, v in filtered_bigrams.items() if word1 in k}).total()
    net.add_node(word1, label=word1, value=word1_tot, title=f"Count: {word1_tot}")
    word2_tot = Counter({k: v for k, v in filtered_bigrams.items() if word2 in k}).total()
    net.add_node(word2, label=word2, value=word2_tot, title=f"Count: {word2_tot}")
    net.add_edge(word1, word2, title=f"Count: {count}", value=count)

# Save and display graph in browser
net.show("bigram_graph.html")

In [7]:
df['description'].iloc[0]

'about workato =================    workato transforms technology complexity into business opportunity. as the leader in enterprise orchestration, workato helps businesses globally streamline operations by connecting data, processes, applications, and experiences. its ai-powered platform enables teams to navigate complex workflows in real-time, driving efficiency and agility.    trusted by a community of 400,000 global customers, workato empowers organizations of every size to unlock new value and lead in today\'s fast-changing world. learn how workato helps businesses of all sizes achieve more at workato.com.   why join us? ================    ultimately, workato believes in fostering a flexible, trust-oriented culture that empowers everyone to take full ownership of their roles. we are driven by innovation and looking for team players who want to actively build our company.    but, we also believe in balancing productivity with self-care. that\'s why we offer all of our employees a v

In [27]:
import torch
from transformers import pipeline
import json

# Load the pre-trained model for zero-shot classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# Define possible labels for classification
candidate_labels = ["company description", "role or task description", "technical expertise or required knowledge"]

for description_text in df['description'].values[4:10]:
    # Split the text into chunks (here using periods as the separator, you can adjust this)
    chunks = description_text.split('.')
    
    # Initialize a dictionary to store results
    classified_text = {
        "company description": [],
        "role or task description": [],
        "technical expertise or required knowledge": []
    }
    
    # Variable to track the current category
    current_category = None
    
    # Process each chunk and classify it
    for chunk in chunks:
        chunk = chunk.strip()  # Clean up leading/trailing spaces
        if not chunk:
            continue
        
        result = classifier(chunk, candidate_labels)
        best_label = result['labels'][0]
        
        # If the category changes, update the current_category and reset the text
        #if current_category != best_label:
        #    current_category = best_label
            
        # Add the chunk to the corresponding category
        classified_text[best_label].append(chunk)
    
    # Convert dictionary to a JSON-like structure (Python dict)
    json_output = json.dumps(classified_text, ensure_ascii=False, indent=2)
    
    # Print the final JSON output
    print(json_output)


Device set to use mps:0


{
  "company description": [],
  "role or task description": [
    "bruker",
    "location: barcelona or zaragoza responsabilidades:  familiarization with specific instrumentation prior to installation from issued documentation and/or participation in the final testing process in the factory  liaise with customers and colleagues in the site planning process prior to installation  inspect, install, set up, test and achieve specifications of systems and accessories at customers’ sites, and deliver basic operator training  carry out breakdown and planned maintenance at customers’ sites  provide technical, software and application related support and assistance to customers and to colleagues, either directly or by remote diagnosis  carry out procedures necessary to validate systems to certifiable standards  provide technical input to the sales team in non-routine sales cases  frequent travel throughout spain is an essential part of the job",
    "travel to customer sites abroad can occur, 

In [22]:
print(f"""Minimun length of description: {min(df['description'].apply(lambda n: len(n.split())))} \n
          maximum: {max(df['description'].apply(lambda n: len(n.split())))}"
        """)

Minimun length of description: 36 

          maximum: 1530"
        


In [23]:
#summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

#for description_text in df['description'].values[4:10]:
#    print(summarizer(description_text, max_length=300, min_length=30, do_sample=False))

In [24]:
oracle = pipeline(model="deepset/roberta-base-squad2")
question_1 = "Which is the field of work or department of the offered position?"
question_2 = "Which are the main responsibilities for the job position described?"
question_3 = "Which are the required technical skills and experience for the job position?"

for description_text in df['description'].values[4:10]:
    print(oracle(question=question_1, context=description_text, max_answer_len=300))
    print(oracle(question=question_2, context=description_text, max_answer_len=300))
    print(oracle(question=question_3, context=description_text, max_answer_len=300))

Device set to use mps:0


{'score': 0.01792718470096588, 'start': 2388, 'end': 2408, 'answer': 'industry or academia'}
{'score': 0.11668083816766739, 'start': 1250, 'end': 1292, 'answer': 'participation in the final testing process'}
{'score': 0.06660237908363342, 'start': 2151, 'end': 2214, 'answer': 'electrical or electronic engineering, chemistry or biochemistry'}
{'score': 0.057732854038476944, 'start': 2381, 'end': 2401, 'answer': 'industry or academia'}
{'score': 0.033846430480480194, 'start': 1142, 'end': 1285, 'answer': 'familiarization with specific instrumentation prior to installation from issued documentation and/or participation in the final testing process'}
{'score': 0.14190460741519928, 'start': 2144, 'end': 2207, 'answer': 'electrical or electronic engineering, chemistry or biochemistry'}
{'score': 0.09230197221040726, 'start': 968, 'end': 983, 'answer': 'project officer'}
{'score': 0.03871304541826248, 'start': 1032, 'end': 1212, 'answer': 'supporting and executing strategic projects within th

In [5]:
from huggingface_hub import login


# Load Hugging Face token
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
# Replace "hf_xxx" with your actual token
login(token=HF_TOKEN)

In [ ]:
import streamlit as st
from transformers import pipeline
import os
import torch

#HF_TOKEN = os.environ["HUGGINGFACE_TOKEN"]
HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

def load_model():
    return pipeline(
        "text-generation",
        model="google/gemma-2-9b-it",#"google/gemma-2b",
        token=HF_TOKEN,
        device=-1
    )

nlp = load_model()

text = "Hello! Are you working now?"
response = nlp(text, max_length=100)
print(response[0]['generated_text'])

model-00002-of-00004.safetensors:  54%|#####4    | 2.68G/4.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

In [ ]:

inputs = tokenizer(input_text, return_tensors="pt")